# Segment 2 - MCP Chat CLI (Lab 02)

**50-minute segment.** The smallest *complete* MCP system: a Python **client**, a stdio FastMCP **server**, and a chat **REPL**. Lab 01 had no client; WARNERCO has too much to read in one sitting. Lab 02 is the Goldilocks middle - the conceptual bridge.

**What you'll do:**
1. Inspect the server's real primitives (2 tools, 1 resource + 1 template, 1 prompt)
2. Compare the two-FastMCP-SDK gotcha (`list_tools()` here vs `get_tools()` in WARNERCO)
3. Run the live chat loop via `run.ps1`

> Anchor cell first. Pick the **Python 3.13** kernel.

## Setup: anchor to the repo root

Same reliability backbone as every segment notebook - walk up to the `.git` marker so paths resolve from any launch directory.

In [1]:
# --- repo-root anchor (identical in all four segment notebooks) ---
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing .git is found."""
    for d in [start, *start.parents]:
        if (d / ".git").exists():
            return d
    raise RuntimeError("Repo root (.git) not found above " + str(start))


REPO = find_repo_root(Path.cwd())
LAB01 = REPO / "labs" / "lab-01-hello-mcp"
LAB02 = REPO / "labs" / "lab-02-mcp-chat"
WARNERCO = REPO / "src" / "warnerco" / "backend"

print("REPO :", REPO)
print("LAB02:", LAB02, "exists:", LAB02.exists())
assert LAB02.exists(), "Lab 02 not found - is REPO correct?"

REPO : C:\github\context-engineering
LAB02: C:\github\context-engineering\labs\lab-02-mcp-chat exists: True


## Introspect the Lab 02 server live

This shells into Lab 02's own venv (`uv run`) and lists its real primitives. **Note the API:** Lab 02 uses the *bundled* `mcp.server.fastmcp.FastMCP`, whose introspection methods are `list_tools()` / `list_resources()` / `list_prompts()` - the low-level MCP names. WARNERCO (Segment 3) uses the *standalone* `fastmcp` package with `get_tools()`. Same protocol, two SDKs - a great teaching point.

In [2]:
# Introspect Lab 02 server in-process via its own venv
probe = r'''
import asyncio
from mcp_server import mcp
async def go():
    t  = await mcp.list_tools()
    r  = await mcp.list_resources()
    rt = await mcp.list_resource_templates()
    p  = await mcp.list_prompts()
    print("tools:    ", [x.name for x in t])
    print("resources:", [str(x.uri) for x in r])
    print("templates:", [x.uriTemplate for x in rt])
    print("prompts:  ", [x.name for x in p])
asyncio.run(go())
'''
r = subprocess.run(["uv", "run", "python", "-c", probe], cwd=LAB02,
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

tools:     ['read_doc_contents', 'edit_document']
resources: ['docs://documents']
templates: ['docs://documents/{doc_id}']
prompts:   ['format']



## Live demo: the chat loop via run.ps1

Lab 02's whole reason to exist is the client + REPL. Run its on-rails launcher in a VS Code terminal:

```powershell
cd labs/lab-02-mcp-chat
.\run.ps1
```

`run.ps1` bootstraps the lab `.env`, lifts `ANTHROPIC_API_KEY` from the repo-root `.env` (no double key paste), and hands off to the REPL.

**If you're presenting this:** "Lab 01 was a server with no brain. Now we add the two missing pieces - a client that speaks MCP, and a model behind it. Type `/format <doc>` and watch the prompt primitive fire. This is the full loop in about 200 lines, before we scale it up to WARNERCO."

> The only working slash command is `/format` (there is no `/summarize` - that was an upstream TODO never implemented).